# Проект 1. Окупилось ли повышение цен в марте 2026 и что оно изменило в маркетинге

**Бизнес-вопрос.** 1 марта 2026 года ПРАЙМ поднял абонентскую плату: Базовый 199 - 249 руб. (+25 %), Расширенный 399 - 449 руб. (+12,5 %).

1. Окупилось ли повышение: сколько выручки оно принесло и чем мы за него заплатили (отток, приток новых подписок, переход на дешевый тариф, неуспешные списания)?
2. Как повышение изменило окупаемость каналов привлечения, и правильно ли распределен маркетинговый бюджет?

## 1. Подключение

In [ ]:
import itertools
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они у всех одинаковые
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
# Общее оформление графиков (акцентный цвет только у того, о чем вывод, остальное серым)
ACCENT = "#C1272D"
GRAY = "#9CA3AF"
DARK = "#374151"
sns.set_theme(style="white", rc={
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#E5E7EB",
})
plt.rcParams["figure.dpi"] = 110

TARIFF = {1: "Базовый", 2: "Расширенный"}


def mln(x: float, digits: int = 2) -> str:
    """Рубли -> строка в млн руб. с запятой: 11325107 -> '11,33'."""
    return f"{x / 1e6:,.{digits}f}".replace(",", " ").replace(".", ",")


def pct(x: float, digits: int = 1) -> str:
    """Доля -> строка в процентах с запятой: 0.0403 -> '4,0 %'."""
    return f"{x * 100:.{digits}f}".replace(".", ",") + " %"

## 2. Разведочный анализ: как я выбрала вопрос

Сначала провела пять разведочных запроса: справочник цен, выручка по месяцам и тарифам, приток и отток подписок, выручка по каналам продаж и расходы на маркетинг по каналам привлечения. Нужно посмотреть, где за весь датасет что-то заметно изменилось

In [ ]:
# 2.1 Справочник цен: когда и как менялась абонентская плата
tariffs = q("""
    select tariff_id, tariff_name, monthly_fee, share_slots, valid_from, valid_to
    from prime.tariffs
    order by tariff_id, valid_from
""")
tariffs

In [ ]:
# 2.2 Выручка по месяцам и тарифам

# Одна строка = один месяц x один тариф

# Откуда данные:
#   prime.payments p       - платежи (p)
#   prime.subscriptions s  - подписки (s)
#   join ... using (subscription_id) соединяет платеж с его подпиской
#   У каждого платежа ровно одна подписка, поэтому строк после соединения столько же сколько платежей
#   subscription_members НЕ беру (двойной счет из-за шеринга, проверка в разделе 3)

# Что считаю в каждой колонке:
#   month       - месяц платежа: to_char превращает момент в строку месяца
#   tariff_id   - тариф: он есть только в подписке, поэтому и нужен join
#   paying_subs - сколько разных (!!) подписок заплатили за месяц (distinct это без повторов)
#   n_success   - сколько списаний прошло
#   n_failed    - сколько списаний не прошло
#   revenue     - выручка: сумма только успешных списаний

# filter (where ...) - посчитать только строки, где условие выполнено (получаю и успешные, и неуспешные списания)



revenue = q("""
    select to_char(p.paid_at, 'YYYY-MM')                                          as month,
           s.tariff_id,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           count(*) filter (where p.status = 'success')                           as n_success,
           count(*) filter (where p.status = 'failed')                            as n_failed,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    group by 1, 2
    order by 1, 2
""")
# group by 1, 2 - сгруппировать по 1-й и 2-й колонке (месяц и тариф)
# order by 1, 2 - отсортировать по месяцу, внутри месяца по тарифу


# Средний чек по тарифу: выручка / число успешных списаний
revenue["avg_check"] = (revenue["revenue"] / revenue["n_success"]).round(2)
revenue.pivot(index="month", columns="tariff_id", values=["revenue", "avg_check"])

In [ ]:
# 2.3 Приток и отток подписок по месяцам

# Одна строка результата = один месяц
# Число подписчиков меняется двумя способами (кто-то оформляет, другие отменяют)

# Запрос состоит из двух частей (with ... as):

# new_subs: сколько подписок оформили за месяц
#   month     - месяц оформления (started_at)
#   new_subs  - всего новых подписок
#   new_trial - из них оформлены в триальный период (is_trial = true)

# cancels: сколько подписок отменили за месяц
#   беру только подписки, у которых есть дата отмены (ended_at is not null)
#   month     - месяц отмены (ended_at)
#   cancelled - всего отмен
#   дальше отмены по причинам:
#     too_expensive  - дорого
#     not_using      - не пользуется
#     payment_failed - не прошла оплата
#     switched       - перешел на другой сервис
#     other          - другое

# В конце соединяю две части по месяцу (!!!)
# full join нужен, чтобы не потерять месяц если в нем были только оформления или только отмены
# Обычный join оставил бы только месяцы, где есть и то и другое
# Если в месяце не было отмен, в колонках отмен будет NaN

flows = q("""
    with new_subs as (
        select to_char(started_at, 'YYYY-MM') as month,
               count(*)                         as new_subs,
               count(*) filter (where is_trial) as new_trial
        from prime.subscriptions
        group by 1
    ),
    cancels as (
        select to_char(ended_at, 'YYYY-MM')                              as month,
               count(*)                                                  as cancelled,
               count(*) filter (where cancel_reason = 'too_expensive')   as too_expensive,
               count(*) filter (where cancel_reason = 'not_using')       as not_using,
               count(*) filter (where cancel_reason = 'payment_failed')  as payment_failed,
               count(*) filter (where cancel_reason = 'switched')        as switched,
               count(*) filter (where cancel_reason = 'other')           as other
        from prime.subscriptions
        where ended_at is not null
        group by 1
    )
    select *
    from new_subs
    full join cancels using (month)
    order by month
""")

flows.tail(12)

In [ ]:
# 2.4 Выручка по каналам продаж по месяцам

# Проверить не влияет ли какой-то один канал на общую выручку сильнее остальных

# Одна строка результата запроса = один месяц x один канал

# Откуда беру данные:
#   prime.payments p       - платежи
#   prime.subscriptions s  - подписки, в них есть номер канала partner_id
#   prime.partners pr      - справочник каналов, в нем есть название канала

# Соединения (строк после соединений столько же, сколько платежей, без двойного счета):
#   платеж в подписку по subscription_id (у платежа ровно одна подписка)
#   подписка в канал по partner_id (у подписки ровно один канал)

# Что считаю:
#   month        - месяц платежа
#   partner_name - название канала
#   paying_subs  - сколько разных подписок заплатили за месяц
#   revenue      - выручка, сумма только успешных списаний

by_partner = q("""
    select to_char(p.paid_at, 'YYYY-MM')                                          as month,
           pr.partner_name,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    join prime.partners pr using (partner_id)
    group by 1, 2
    order by 1, 2
""")

# Разворачиваю таблицу через (pivot):
# месяцы по строкам, каналы по столбцам, в ячейках выручка

by_partner.pivot(index="month", columns="partner_name", values="revenue").tail(8)

In [ ]:
# 2.5 Маркетинг: сколько тратят на каждый канал привлечения и сколько новых подписок он приносит

# Канал беру у владельца подписки (clients.acquisition_channel), расходы из marketing_spend
# Окно: вся история расходов, с 6 января 2025 по 29 августа 2026
# CAC = расходы канала / новые подписки клиентов этого канала за то же время 

# spend - расходы по каналам за всю историю
# subs  - новые подписки по каналу владельца за те же даты; пустой канал заменяю на "без канала"
# left join от подписок: канал без расходов не потеряется, у него будут нулевые расходы
marketing = q("""
    with spend as (
        select channel, sum(amount) as spend
        from prime.marketing_spend
        group by 1
    ),
    subs as (
        select coalesce(c.acquisition_channel, 'без канала') as channel, count(*) as new_subs
        from prime.subscriptions s
        left join prime.clients c on c.client_id = s.owner_client_id
        where s.started_at >= '2025-01-06' and s.started_at < '2026-08-30'
        group by 1
    )
    select sb.channel, coalesce(sp.spend, 0) as spend, sb.new_subs
    from subs sb
    left join spend sp using (channel)
    order by 2 desc
""")

# Доля канала в бюджете и CAC; у категории без канала расходов нет, поэтому CAC не считаю
marketing["доля бюджета"] = (marketing["spend"] / marketing["spend"].sum()).round(3)
marketing["CAC"] = (marketing["spend"] / marketing["new_subs"]).where(marketing["spend"] > 0).round(0)
marketing

### Что было найдено в разведке


1. 1 марта 2026 изменились цены. В справочнике Базовый, который стоил 199 стал 249 руб., Расширенный из 399 в 449 руб. По платежам видно то же самое: средний чек до февраля 2026 ровно 199 и 399, с марта 249 и 449. Оба тарифа подорожали на 50 руб., но в процентах по-разному: +25 % и +12,5 % (потому что базовый стоит дешевле)
2. В марте 2026 выручка выросла примерно на треть к февралю. Это самый большой скачок за всю историю платежей
3. Провалы в феврале и апреле не похожи на проблему. В феврале 28 дней, в апреле 30, а списание идет раз в 30 дней. Месяцы разной длины необходимо сравнивать только в пересчете на день. В апреле 2026 также есть пропуск в выгрузке: за 14 и 15 апреля нет платежей
4. История платежей неполная. Платежи начинаются 6 января 2025 и заканчиваются 30 августа 2026. Кроме того, в августе 2026 отмен в несколько раз больше обычного: это пакетная отмена подписок partner_telecom 22-23 августа. Так как к повышению цен она отношения не имеет, январь 2025 и август 2026 в анализ не буду брать
5. Маркетинговый бюджет распределен неравномерно. Половина расходов уходит в performance, и подписка через него обходится примерно в 11 раз дороже, чем через organic (2 177 и 196 руб., то же было видно на семинаре 5). Дорогой канал должен окупаться лучше остальных, иначе бюджет распределен неудачно (отсюда гипотезы Г10-Г11)
6. Выручка по семи каналам продаж почти одинаковая, и все каналы растут примерно одинаково (проверка будет ниже)
7. В справочнике тарифов есть подозрительная строка «Расширенный 429 руб., 01.02.2026-30.06.2026». Ее период пересекается с периодами двух соседних строк (эту ситуацию разберу в разделе 3)
8. Новых подписок каждый месяц примерно одинаково, около 11 тыс., а отмен с каждым месяцем больше: около 4 тыс. в январе 2025 и около 7,4 тыс. в июле 2026. Около четверти отмен идет с причиной «дорого», а неуспешных списаний около 3,4 % попыток. После повышения цен любая из этих цифр могла ухудшиться (отсюда гипотезы Г3-Г7)

Отсюда и возникает главный вопрос проекта: повышение цен стало единственным крупным изменением за весь период, но его последствия еще не анализировали. При этом новые цены влияют не только на выручку с подписки, но и на окупаемость маркетинга (это будут гипотезы 10-11)

### Гипотезы

| | Гипотеза | Чем проверяю | Что будет, если гипотеза верна |
|---|---|---|---|
| Г1 | Прирост выручки дал один канал продаж | аддитивная модель по каналам | один канал заберет большую часть прироста |
| Г2 | Прирост выручки в марте дала цена, а не рост базы | мультипликативная модель: активные x платежей на подписку x цена | влияние цены больше влияния базы |
| Г3 | Повышение увеличило отток, особенно по причине «дорого» | отток до и после, тот же сезон год назад, сравнение тарифов | отток после марта выше; у Базового (+25 %) рост больше, чем у Расширенного (+12,5 %) |
| Г4 | Повышение снизило приток новых подписок | новые подписки в день до и после | приток после марта ниже |
| Г5 | Клиенты стали выбирать дешевый тариф | доля Расширенного среди новых и среди активных | доля Расширенного падает |
| Г6 | Стало больше неуспешных списаний (сумма выросла, на карте не хватает денег) | доля failed до и после | доля failed растет |
| Г7 | После повышения замедлился рост базы | чистый прирост базы по месяцам и тренд до повышения | прирост после марта ниже прежнего тренда |
| Г8 | Отток различается по каналам привлечения | отток каналов до и после | каналы устойчиво различаются |
| Г9 | Повышение сильнее ударило по каналам, чувствительным к цене | изменение оттока каждого канала и его 95 % ДИ | у какого-то канала отток заметно вырос |
| Г10 | Дорогой канал (performance) приводит более ценных клиентов | ARPPU, доля Расширенного и отток performance против остальных | у performance выше ARPPU или ниже отток |
| Г11 | Новая цена сделала все каналы окупаемыми | LTV/CAC каналов до и после, разложение изменения на факторы | у всех каналов LTV/CAC после повышения не ниже 3 |

Каналы продаж (Г1) и каналы привлечения (Г8-Г11) это разные признаки: первый у подписки (`subscriptions.partner_id`, через кого оформили), второй у клиента (`clients.acquisition_channel`, откуда пришел, на него идут маркетинговые расходы).

In [ ]:
# Г1. Раскладываю прирост выручки январь - март 2026 по каналам продаж (аддитивная модель)
# Беру сырые успешные платежи из 2.4: двойные списания дают около 0,1 % выручки и на доли каналов не влияют (чистая выручка считается в разделе 5)
# Январь и март по 31 дню, поэтому длина месяца сравнению не мешает
# Выручка равна сумме выручек каналов (ее изменение равно сумме изменений каналов)
channels = by_partner.pivot(index="partner_name", columns="month", values="revenue")
channels = channels[["2026-01", "2026-03"]].copy()
channels["delta"] = channels["2026-03"] - channels["2026-01"]
channels["growth"] = channels["2026-03"] / channels["2026-01"] - 1
channels["share_of_delta"] = channels["delta"] / channels["delta"].sum()

# Сверка: сумма изменений каналов равна изменению итога 
total_jan = revenue.loc[revenue["month"] == "2026-01", "revenue"].sum()
total_mar = revenue.loc[revenue["month"] == "2026-03", "revenue"].sum()
assert abs(channels["delta"].sum() - (total_mar - total_jan)) < 0.01, "каналы не сходятся с итогом"

print("рост выручки по каналам: от", pct(channels["growth"].min()), "до", pct(channels["growth"].max()))
channels.sort_values("delta").round({"delta": 0, "growth": 4, "share_of_delta": 4})

**Г1 не подтвердилась.** Прирост распределен по семи каналам почти поровну, у каждого около 1/7 общего прироста, и растут они одинаковым темпом. Значит причина общая для всех каналов, и дальше канал продаж как сегмент не нужен. Сегментом будет тариф, так как цена менялась именно по тарифам, и по-разному в процентах.

## 3. Проверки данных до расчета выручки

8 проверок, чтобы понять, как можно считать выручку

In [ ]:
# 3.1 Какой период покрывают платежи
span = q("""
    select min(paid_at)::text as first_payment,
           max(paid_at)::text as last_payment,
           count(*)           as n_payments
    from prime.payments
""")
span

In [ ]:
# 3.2 Платежи без подписки 
# Внешних ключей в базе нет (платеж может ссылаться на несуществующую подписку)
# left join оставляет все платежи; где подписка не нашлась там s.subscription_id пустой
orphans = q("""
    select count(*)                                                    as payments_without_subscription,
           coalesce(sum(p.amount) filter (where p.status = 'success'), 0) as their_revenue
    from prime.payments p
    left join prime.subscriptions s using (subscription_id)
    where s.subscription_id is null
""")
orphans

In [ ]:
# Сверяю число строк до и после соединения платежей с подписками
# subscription_id в subscriptions уникален, поэтому платежи размножиться не могут,
# а потеряться могут только платежи без подписки
rows_check = q("""
    select (select count(*) from prime.payments)                                   as rows_payments,
           (select count(*) from prime.payments p
                join prime.subscriptions s using (subscription_id))                as rows_after_join,
           (select count(*) - count(distinct subscription_id) from prime.subscriptions) as duplicate_subscription_ids
""")
rows_check["lost_in_join"] = rows_check["rows_payments"] - rows_check["rows_after_join"]
assert rows_check.loc[0, "duplicate_subscription_ids"] == 0, "subscription_id не уникален: join размножит платежи"
assert rows_check.loc[0, "lost_in_join"] == orphans.loc[0, "payments_without_subscription"], "join теряет не только сирот"
rows_check

In [ ]:
# 3.3 Двойной счет из-за шеринга на примере июля 2026

# Платит владелец подписки, он записан в subscriptions
# В subscription_members лежат только те, с кем поделились (без владельца) (у трети подписок там нет ни одной строки)

# Если соединить платежи с участниками:
#   1) платеж повторится столько раз, сколько участников в подписке (выручка раздуется)
#   2) подписки без участников выпадут из inner join (выручка потеряется)

# Сравниваю число строк и выручку до и после такого соединения
sharing = q("""
    with p as (
        select subscription_id, amount
        from prime.payments
        where status = 'success'
          and paid_at >= '2026-07-01' and paid_at < '2026-08-01'
    )
    select 'правильно: платеж -> подписка'::text as how, count(*) as n_rows, sum(amount) as revenue
    from p
    union all
    select 'ошибка: платеж -> участники подписки', count(*), sum(p.amount)
    from p
    join prime.subscription_members m using (subscription_id)
    union all
    select 'из них выпали: подписки без участников', count(*), sum(amount)
    from p
    where not exists (select 1 from prime.subscription_members m where m.subscription_id = p.subscription_id)
""")
sharing["vs_correct"] = sharing["revenue"] / sharing.loc[0, "revenue"]
sharing.round({"vs_correct": 3})

In [ ]:
# 3.4 Справочник тарифов (чтобы не было пересекающихся интервалов цены)

# Соединяю справочник (tariff_id, чтобы вторая строка начиналась раньше,чем закончилась первая
#В ок справочнике таких пар не должно быть (!)
overlaps = q("""
    select a.tariff_id,
           a.monthly_fee as fee_1, a.valid_from as from_1, a.valid_to as to_1,
           b.monthly_fee as fee_2, b.valid_from as from_2, b.valid_to as to_2
    from prime.tariffs a
    join prime.tariffs b
      on a.tariff_id = b.tariff_id
     and a.valid_from < b.valid_from
     and b.valid_from <= a.valid_to
    order by 1, 3, 6
""")
overlaps

In [ ]:
# Что будет, если соединить платежи с ценой по интервалу (из базы)
# Беру успешные платежи января - июля 2026 и считаю строки до и после соединения
tariff_join = q("""
    with p as (
        select p.payment_id, p.paid_at, s.tariff_id
        from prime.payments p
        join prime.subscriptions s using (subscription_id)
        where p.status = 'success'
          and p.paid_at >= '2026-01-01' and p.paid_at < '2026-08-01'
    )
    select (select count(*) from p) as rows_before,
           (select count(*)
            from p
            join prime.tariffs t
              on t.tariff_id = p.tariff_id
             and cast(p.paid_at as date) between t.valid_from and t.valid_to) as rows_after
""")
tariff_join["extra_rows"] = tariff_join["rows_after"] - tariff_join["rows_before"]
tariff_join

In [ ]:
# Какие суммы на самом деле списывали в 2026 году
# Если строка 429 руб. реальная, в феврале - июне у расширенного должны быть платежи по 429
charged = q("""
    select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, p.amount, count(*) as n
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    where p.status = 'success'
      and p.paid_at >= '2026-01-01' and p.paid_at < '2026-08-01'
    group by 1, 2, 3
    order by 1, 2, 3
""")
charged.pivot(index="month", columns=["tariff_id", "amount"], values="n").fillna(0).astype(int)

In [ ]:
# 3.5 проверяю двойные списания (два успешных платежа одной подписки на одну сумму с разницей меньше 20 минут, второй - дубль)
# lag(paid_at) over (partition by subscription_id, amount order by paid_at)- момент предыдущего успешного платежа той же подписки на ту же сумму
# gap - сколько прошло с него

# зависит ли ответ от порога 
dup_threshold = q("""
    with ok as (
        select paid_at - lag(paid_at) over (partition by subscription_id, amount order by paid_at) as gap
        from prime.payments
        where status = 'success'
    )
    select count(*) filter (where gap < interval '1 minute')   as "1 мин",
           count(*) filter (where gap < interval '5 minutes')  as "5 мин",
           count(*) filter (where gap < interval '20 minutes') as "20 мин",
           count(*) filter (where gap < interval '1 hour')     as "1 час",
           count(*) filter (where gap < interval '1 day')      as "1 день",
           count(*) filter (where gap < interval '7 days')     as "7 дней"
    from ok
""")
dup_threshold

In [ ]:
# Дубли по месяцам (сколько и на сколько они завышают выручку)
dups = q("""
    with ok as (
        select subscription_id, amount, paid_at,
               paid_at - lag(paid_at) over (partition by subscription_id, amount order by paid_at) as gap
        from prime.payments
        where status = 'success'
    )
    select to_char(paid_at, 'YYYY-MM')                                               as month,
           count(*) filter (where gap < interval '20 minutes')                       as n_dup,
           count(distinct subscription_id) filter (where gap < interval '20 minutes') as dup_subs,
           coalesce(sum(amount) filter (where gap < interval '20 minutes'), 0)       as dup_revenue,
           sum(amount)                                                               as revenue_raw
    from ok
    group by 1
    order by 1
""").set_index("month")
dups["dup_share"] = dups["dup_revenue"] / dups["revenue_raw"]

print("всего дублей:", int(dups["n_dup"].sum()),
      "| завышение выручки за всю историю:", pct(dups["dup_revenue"].sum() / dups["revenue_raw"].sum(), 3))
dups.tail(8).round({"dup_share": 5})

In [ ]:
# 3.6 Возвраты (status = 'refunded')
# насколько велики возвраты и не изменились ли они после повышения (если да, их вычту)
refunds = q("""
    select to_char(paid_at, 'YYYY-MM')                                    as month,
           count(*) filter (where status = 'refunded')                    as n_refunded,
           coalesce(sum(amount) filter (where status = 'refunded'), 0)    as refunded,
           sum(amount) filter (where status = 'success')                  as revenue_raw
    from prime.payments
    group by 1
    order by 1
""").set_index("month")
refunds["refund_share"] = refunds["refunded"] / refunds["revenue_raw"]

print("возвраты от выручки: до повышения (09.2025-02.2026)", pct(refunds.loc["2025-09":"2026-02", "refund_share"].mean(), 2),
      "| после (03-07.2026)", pct(refunds.loc["2026-03":"2026-07", "refund_share"].mean(), 2))
refunds.tail(8).round({"refund_share": 5})

In [ ]:
# 3.7 Дыры в данных (eg есть ли дни, за которые нет ни одного платежа)
# generate_series дает все дни истории, left join оставляет и те, где платежей нет
missing_days = q("""
    with days as (
        select cast(generate_series(date '2025-01-06', date '2026-08-30', interval '1 day') as date) as d
    ),
    have as (
        select distinct cast(paid_at as date) as d
        from prime.payments
    )
    select days.d::text as day_without_payments
    from days
    left join have using (d)
    where have.d is null
    order by 1
""")
print("дней без единого платежа:", len(missing_days))
missing_days

In [ ]:
# Сколько дней с данными в каждом месяце (выручку в день дальше делю на дни, а не на календарь)
days_with_data = q("""
    select to_char(d, 'YYYY-MM') as month, count(*) as days_with_data
    from (select distinct cast(paid_at as date) as d from prime.payments) x
    group by 1
    order by 1
""").set_index("month")["days_with_data"]
days_with_data.tail(8)
# иначе месяц с дырой выглядел бы как падение

In [ ]:
# 3.8 Сколько подписок бывает у одного клиента

subs_per_owner = q("""
    select sum(n)   as subscriptions,
           count(*) as owners,
           max(n)   as max_per_owner
    from (select owner_client_id, count(*) as n from prime.subscriptions group by 1) x
""")
subs_per_owner

# если у клиента не больше одной подписки, то новая подписка канала = новый подписавшийся клиент и CAC на подписку считается без двойного счета

### Итоги проверок

| Проверка | Результат | Решение |
|---|---|---|
| Период | платежи с 6 января 2025 по 30 августа 2026 | берем полные месяцы: февраль 2025 - июль 2026 |
| Платежи без подписки | число и выручка сирот в таблице 3.2; join с подписками не размножает строк (subscription_id уникален) и теряет ровно платежей без подписки, это проверено `assert` | платежи без подписки не относятся ни к одному тарифу, поэтому в выручку по тарифам не входят; итоги витрины сверяю с разведкой (раздел 5) |
| Шеринг | соединение с `subscription_members` раздувает выручку и одновременно теряет подписки без участников | считаю от платежа к подписке: одна подписка = один плательщик |
| Справочник тарифов | строка 429 руб. пересекается с двумя соседними, соединение по интервалу добавляет лишние строки, а платежей по 429 руб. не было ни одного | справочник с платежами не соединяем: фактическая цена есть в самом платеже (`amount`) |
| Двойные списания | ответ не меняется в широком диапазоне порогов, дубли завышают выручку примерно на десятую долю процента | дубли из выручки исключаем, порог 20 минут |
| Дыры в данных | за 14 и 15 апреля 2026 нет ни одного платежа | выручку в день считаю по дням с данными; апрель 2026 в декомпозицию не беру (окна: январь, март, июль) |
| Подписок на клиента | см. таблицу 3.8 | если у клиента не больше одной подписки, CAC на подписку равен CAC на подписавшегося клиента |
| Возвраты | доля возвратов в выручке маленькая и одинаковая до и после повышения | выручку считаем по успешным списаниям; на сравнение до и после возвраты не влияют |

Важно: строку 429 руб стоит показать владельцу справочника: это скорее всего ошибка данных, и любой отчет, который соединяет платежи с ценой по датам, сейчас завышает выручку расширенного тарифа с февраля по июнь 2026.

## 4. Зерно, период и сегменты

**Зерно витрины: месяц x тариф.** В каждой строке: активные подписки, новые, ушедшие, число успешных платежей, выручка, неуспешные списания

- Почему считаю подписки, а не клиентов? Абонентскую плату вносит владелец подписки, а участники шеринга ничего не платят. Если считать по людям из subscription_members, одна и та же выручка может учесться несколько раз. В разделе 3.8 проверю, сколько подписок приходится на одного владельца

- Почему не платежи? Мне важно понять, сколько подписок у нас есть, сколько из них прекращают действовать и сколько продолжают приносить выручку. По одним только платежам этого не понять, так их количество может вырасти как из-за увеличения числа подписок, так и из-за более частых списаний

- Почему считаю по месяцам? Списание происходит раз в 30 дней, поэтому за месяц каждая активная подписка обычно оплачивается примерно один раз. Дневные данные слишком нестабильны, а цены изменились ровно с первого числа месяца. Разницу в количестве дней учитываю отдельно (для декомпозиции сравниваю месяцы одинаковой длины, а на графиках показываю дневную выручку)

- Почему разделяю подписки по тарифам? Цены на тарифы выросли по-разному именно в процентах: на 25% для «Базового» и на 12,5% для «Расширенного». Это позволяет сравнить реакцию пользователей на повышение. Если клиенты чувствительны к цене, на «Базовом» эффект должен быть заметнее

- Что считаю активной подпиской? По определению из лекции 5, подписка считается активной в месяце, если она началась до его окончания и не закончилась до его начала

- Как считаю отток? По определению из семинара 5, это доля подписок, которые были активны в начале месяца, но закончились до его конца. Подписки, оформленные в течение этого месяца, в расчёт не включаю

**Зерно для маркетинга (Г8-Г11): канал привлечения x окно.** Строка витрины: месяц x канал, затем месяцы окна складываются. Почему так:

* Канал берем у владельца подписки (`clients.acquisition_channel`): маркетинговые расходы ведутся по этим же каналам, а платит владелец. Клиенты без канала идут отдельной строкой; расходов на них нет, поэтому CAC и LTV/CAC у них не считаются 
* Отток канала за один месяц шумный (на канал приходится 3-60 тыс. подписок), поэтому складываю все месяцы окна до и после: отток = все ушедшие / все активные на 1е число
* Определения с семинара 5: CAC = расходы канала за окно / новые подписки канала за окно; ARPPU = выручка / платящие подписки (по месяцам окна); LTV = ARPPU / отток; канал окупается, если LTV/CAC не ниже 3. LTV здесь по выручке и по формуле, это прогноз при неизменных оттоке и платеже

**Периоды.**

* Вся история для графиков: февраль 2025 - июль 2026. Январь 2025 неполный (платежи с 6 января), август 2026 неполный и испорчен пакетной отменой partner_telecom
* Декомпозиция: январь 2026 по март 2026. Январь последний месяц по старым ценам с той же длиной, что у марта (31 день). Февраль короче на 3 дня. Для контекста та же декомпозиция за год: июль 2025 по июль 2026 (оба по 31 дню).
* Сравнение до и после для гипотез: до = сентябрь 2025 - февраль 2026 (6 месяцев), после = март - июль 2026 (5 месяцев), сезонный контроль = те же месяцы год назад, март - июль 2025.

## 5. Витрина месяц x тариф

In [ ]:
# Одна строка = месяц x тариф

# months - список первых чисел месяцев от :first до :last (generate_series)

# act - база подписок за месяц (одна подписка попадает в каждый месяц, когда была активна):
#   active            - активна в месяце: началась до конца месяца и не закончилась до его начала
#   new_subs          - оформлена в этом месяце
#   active_start      - активна на 1-е число (началась раньше месяца)
#   churned           - из активных на 1-е число закончилась до конца месяца
#   churned_expensive - из них с причиной too_expensive ("дорого")
#   churned_switched  - из них с причиной switched ("перешел")

# ok - успешные платежи и gap до предыдущего успешного платежа той же подписки на ту же сумму
# rev - выручка без двойных списаний (gap меньше 20 минут = дубль, раздел 3.5):
#   n_payments - число успешных платежей
#   n_paying   - сколько разных подписок заплатили (для ARPPU)
#   revenue    - выручка
# fail - неуспешные списания

# В конце база соединяется с выручкой по месяцу и тарифу
# left join: если в месяце у тарифа нет платежей, строка базы не потеряется (будет 0)
FIRST, LAST = "2025-02-01", "2026-07-01"

vitrina = q("""
    with months as (
        select generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m
    ),
    act as (
        select to_char(mo.m, 'YYYY-MM') as month,
               s.tariff_id,
               count(*)                                         as active,
               count(*) filter (where s.started_at >= mo.m)     as new_subs,
               count(*) filter (where s.started_at < mo.m)      as active_start,
               count(*) filter (where s.started_at < mo.m
                                  and s.ended_at < mo.m + interval '1 month') as churned,
               count(*) filter (where s.started_at < mo.m
                                  and s.ended_at < mo.m + interval '1 month'
                                  and s.cancel_reason = 'too_expensive') as churned_expensive,
               count(*) filter (where s.started_at < mo.m
                                  and s.ended_at < mo.m + interval '1 month'
                                  and s.cancel_reason = 'switched') as churned_switched
        from months mo
        join prime.subscriptions s
          on s.started_at < mo.m + interval '1 month'
         and (s.ended_at is null or s.ended_at >= mo.m)
        group by 1, 2
    ),
    ok as (
        select subscription_id, amount, paid_at,
               paid_at - lag(paid_at) over (partition by subscription_id, amount order by paid_at) as gap
        from prime.payments
        where status = 'success'
    ),
    rev as (
        select to_char(ok.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               count(*)                           as n_payments,
               count(distinct ok.subscription_id) as n_paying,
               sum(ok.amount)                     as revenue
        from ok
        join prime.subscriptions s using (subscription_id)
        where (ok.gap is null or ok.gap >= interval '20 minutes')
          and ok.paid_at >= cast(:first as timestamptz)
          and ok.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    ),
    fail as (
        select to_char(p.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               count(*) as n_failed
        from prime.payments p
        join prime.subscriptions s using (subscription_id)
        where p.status = 'failed'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    )
    select a.*,
           coalesce(r.n_payments, 0) as n_payments,
           coalesce(r.n_paying, 0)   as n_paying,
           coalesce(r.revenue, 0)    as revenue,
           coalesce(f.n_failed, 0)   as n_failed
    from act a
    left join rev r using (month, tariff_id)
    left join fail f using (month, tariff_id)
    order by 1, 2
""", first=FIRST, last=LAST)

m = vitrina.set_index(["month", "tariff_id"])
print(len(m), "строк: по 2 тарифа на каждый из", m.index.get_level_values("month").nunique(), "месяцев")

In [ ]:
# Сверка витрины с разведкой: выручка витрины = выручка из 2.2 минус двойные списания
# Если сходится, при соединениях ничего не потерялось и не задвоилось
months_in = m.index.get_level_values("month").unique()
raw = revenue.groupby("month")["revenue"].sum().loc[months_in]
clean = m.groupby("month")["revenue"].sum()
gap = raw - dups.loc[months_in, "dup_revenue"] - clean
assert gap.abs().max() < 0.01, f"витрина не сходится с разведкой: {gap.abs().max():.2f} руб."
print("сверка пройдена: выручка витрины = выручка разведки - дубли, во всех", len(months_in), "месяцах")

In [ ]:
# Метрики на уровне месяц x тариф
m["days"] = [pd.Period(month, freq="M").days_in_month for month in m.index.get_level_values("month")]
m["per_sub"] = m["n_payments"] / m["active"]          # платежей на одну активную подписку
m["price"] = m["revenue"] / m["n_payments"]           # средняя цена одного платежа
m["churn"] = m["churned"] / m["active_start"]          # отток месяца
m["churn_expensive"] = m["churned_expensive"] / m["active_start"]
m["fail_rate"] = m["n_failed"] / (m["n_failed"] + m["n_payments"])
m["new_per_day"] = m["new_subs"] / m["days"]

# Те же метрики по всей базе (оба тарифа вместе)
tot = m.groupby("month")[["active", "new_subs", "active_start", "churned", "churned_expensive",
                          "churned_switched", "n_payments", "n_paying", "revenue", "n_failed"]].sum()
tot["days"] = [pd.Period(month, freq="M").days_in_month for month in tot.index]
tot["days_with_data"] = days_with_data.loc[tot.index]
tot["revenue_per_day"] = tot["revenue"] / tot["days_with_data"]   # на дни, за которые есть платежи (дыра 14-15.04.2026)
tot["arpu"] = tot["revenue"] / tot["active"]           # ARPU: выручка на активную подписку
tot["arppu"] = tot["revenue"] / tot["n_paying"]        # ARPPU: выручка на платящую подписку
tot["churn"] = tot["churned"] / tot["active_start"]
tot["churn_expensive"] = tot["churned_expensive"] / tot["active_start"]
tot["churn_switched"] = tot["churned_switched"] / tot["active_start"]
tot["fail_rate"] = tot["n_failed"] / (tot["n_failed"] + tot["n_payments"])
tot["new_per_day"] = tot["new_subs"] / tot["days"]
tot["ext_share_new"] = m.xs(2, level="tariff_id")["new_subs"] / tot["new_subs"]
tot["ext_share_active"] = m.xs(2, level="tariff_id")["active"] / tot["active"]

tot[["active", "revenue", "days", "days_with_data", "revenue_per_day", "arpu", "churn", "new_per_day", "ext_share_new", "fail_rate"]].round(4)

In [ ]:
# График 1. Выручка по месяцам и выручка в день
# Левая панель: как выручку обычно показывают (по месяцам). Правая: она же в пересчете на день

# Для seaborn одна строка = один месяц
# period = до / после повышения: по нему seaborn красит линию (hue) в серый и акцентный цвет
PALETTE = {"до": GRAY, "после": ACCENT}

plot1 = tot[["revenue", "revenue_per_day", "days", "days_with_data"]].reset_index()
plot1["date"] = pd.PeriodIndex(plot1["month"], freq="M").to_timestamp()
plot1["period"] = ["после" if d >= pd.Timestamp("2026-03-01") else "до" for d in plot1["date"]]
plot1["revenue_mln"] = plot1["revenue"] / 1e6
plot1["per_day_mln"] = plot1["revenue_per_day"] / 1e6

jump = tot.loc["2026-03", "revenue_per_day"] / tot.loc["2026-02", "revenue_per_day"] - 1
growth_before = (tot.loc["2026-02", "revenue_per_day"] / tot.loc["2025-09", "revenue_per_day"]) ** (1 / 5) - 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")

ax = axes[0]
sns.lineplot(data=plot1, x="date", y="revenue_mln", color=GRAY, marker="o", ax=ax)
short = plot1[plot1["month"].isin(["2026-02", "2026-04"])]      # месяцы, которые выглядят как провал
sns.scatterplot(data=short, x="date", y="revenue_mln", color=DARK, s=60, ax=ax, zorder=3)
for row in short.itertuples():
    label = f"{row.days} дн." if row.days == row.days_with_data else f"{row.days} дн., {row.days - row.days_with_data} без данных"
    ax.annotate(label, (row.date, row.revenue_mln), xytext=(0, -16),
                textcoords="offset points", ha="center", color=DARK)
ax.set(xlabel="", ylabel="млн руб. за месяц")
ax.set_title("По месяцам: февраль и апрель выглядят как падение", loc="left", fontsize=11)
# В феврале 28 дней, в апреле 30 и еще 2 дня без выгрузки: отсюда "провалы"
ax.set_ylim(bottom=0)

ax = axes[1]
sns.lineplot(data=plot1, x="date", y="per_day_mln", hue="period", palette=PALETTE,
             marker="o", legend=False, ax=ax)
ax.axvline(pd.Timestamp("2026-03-01"), color=ACCENT, linestyle=":", linewidth=1)
ax.annotate("1 марта 2026:\n199 -> 249 и 399 -> 449 руб.",
            (pd.Timestamp("2026-03-01"), tot.loc["2026-03", "revenue_per_day"] / 1e6),
            xytext=(0.10, 0.20), textcoords="axes fraction", color=ACCENT,
            arrowprops={"arrowstyle": "->", "color": ACCENT})
ax.set(xlabel="", ylabel="млн руб. в день")
ax.set_title(f"В день: провалов нет, в марте скачок на {pct(jump, 0)}", loc="left", fontsize=11)
ax.set_ylim(bottom=0)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))

fig.suptitle(f"Выручка в день росла ровно (около {pct(growth_before)} в месяц), а 1 марта 2026 выросла скачком на {pct(jump, 0)}",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.text(0.01, -0.04, "Успешные платежи без двойных списаний, сутки по UTC. Февраль 2025 - июль 2026: январь 2025 и август 2026 неполные. "
         "Выручка в день = выручка / дни с данными (14-15.04.2026 платежей нет).",
         fontsize=9, color=DARK)
plt.show()

**Что доказывает график 1.** В феврале и апреле месячная выручка снижается только потому, что в этих месяцах меньше дней. Если пересчитать ее на день, график получается гораздо ровнее. Единственное заметное изменение за весь период приходится на 1 марта 2026 года, когда повысили цены. Поэтому имеет смысл оценить эффект повышения и при сравнении месячной выручки учитывать разницу в количестве дней

## 6. Факторная декомпозиция: январь 2026 - март 2026

Модели сочетаю: сложение по тарифам, умножение внутри тарифа.

> выручка = сумма по тарифам ( активные подписки x платежей на подписку x цена платежа )

* активные подписки - размер базы;
* платежей на подписку - как часто активная подписка платит за месяц (зависит от длины месяца и неуспешных списаний);
* цена платежа - средняя сумма успешного платежа. Внутри тарифа это и есть цена тарифа.

Метод цепных подстановок, порядок: сначала активные, потом платежей на подписку, потом цена. Структура тарифов (сдвиг долей Базового и Расширенного) при сложении по тарифам попадает в активные подписки каждого тарифа.

In [ ]:
FACTORS = ["active", "per_sub", "price"]
LABELS = {"active": "активные подписки", "per_sub": "платежей на подписку", "price": "цена платежа"}


def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    """Цепные подстановки (семинар 6): меняем факторы по одному в порядке order."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]                       # один фактор переходит на новое значение, остальные стоят
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


def decompose(a: str, b: str, order: list[str] = FACTORS) -> pd.DataFrame:
    """Влияние факторов по тарифам при переходе от месяца a к месяцу b, руб."""
    rows = {}
    for t in TARIFF:
        x0 = m.loc[(a, t), FACTORS].to_dict()
        x1 = m.loc[(b, t), FACTORS].to_dict()
        rows[TARIFF[t]] = contrib(x0, x1, order)
    eff = pd.DataFrame(rows).T[FACTORS]
    # Сумма влияний по всем тарифам должна совпасть с изменением выручки
    delta = tot.loc[b, "revenue"] - tot.loc[a, "revenue"]
    assert abs(eff.to_numpy().sum() - delta) < 0.01, "сумма влияний не равна изменению выручки"
    return eff


A, B = "2026-01", "2026-03"
print("дней в месяцах:", tot.loc[A, "days"], "и", tot.loc[B, "days"])
m.loc[[A, B], ["active", "per_sub", "price", "revenue"]].round(4)

In [ ]:
eff = decompose(A, B)
delta = tot.loc[B, "revenue"] - tot.loc[A, "revenue"]

table = eff.rename(columns=LABELS).copy()
table["всего"] = table.sum(axis=1)
table.loc["итого"] = table.sum()

price_effect = eff["price"].sum()
base_effect = eff["active"].sum()
print(f"выручка: {mln(tot.loc[A, 'revenue'])} -> {mln(tot.loc[B, 'revenue'])} млн руб., изменение {mln(delta)} млн руб.")
print(f"цена дала {mln(price_effect)} млн руб. ({pct(price_effect / delta, 0)} прироста), "
      f"рост базы {mln(base_effect)} млн руб. ({pct(base_effect / delta, 0)})")
(table / 1e6).round(3)

In [ ]:
# Проверяю, меняется ли вывод при другом порядке подстановок (всего 6 порядков)
orders = {}
for order in itertools.permutations(FACTORS):
    e = decompose(A, B, list(order)).sum()
    orders[" -> ".join(LABELS[f] for f in order)] = e
orders = pd.DataFrame(orders).T.rename(columns=LABELS)
orders.loc["среднее по порядкам (Шепли)"] = orders.mean()
(orders / 1e6).round(3)

Порядок подстановок меняет только совместный член (рост базы, умноженный на рост цены). Во всех шести порядках цена остается главным фактором, поэтому вывод от порядка не зависит.

In [ ]:
# Эффект структуры (средняя цена платежа по всей базе смешивает цены и доли тарифов в платежах)
#Проверяю, не вырос ли средний чек еще и за счет того,
# что клиенты стали чаще брать дорогой тариф

#   средняя цена = сумма по тарифам (доля тарифа в платежах x цена тарифа)
#   влияние цен тарифов = сумма (доля после x (цена после - цена до))
#   влияние структуры   = сумма ((доля после - доля до) x цена до)
share = {mo: m.loc[mo, "n_payments"] / m.loc[mo, "n_payments"].sum() for mo in (A, B)}
price = {mo: m.loc[mo, "price"] for mo in (A, B)}
avg_price = {mo: (share[mo] * price[mo]).sum() for mo in (A, B)}
by_tariff_prices = (share[B] * (price[B] - price[A])).sum()
by_mix = ((share[B] - share[A]) * price[A]).sum()
assert abs(by_tariff_prices + by_mix - (avg_price[B] - avg_price[A])) < 1e-6, "структура + цены != изменению среднего чека"

print(f"средний платеж: {avg_price[A]:.2f} -> {avg_price[B]:.2f} руб.")
print(f"  за счет цен тарифов: {by_tariff_prices:+.2f} руб.")
print(f"  за счет структуры (доли тарифов): {by_mix:+.2f} руб.")
pd.DataFrame({"доля в платежах, январь": share[A], "доля в платежах, март": share[B],
              "цена, январь": price[A], "цена, март": price[B]}).rename(index=TARIFF).round(4)

Средний чек вырос почти полностью за счёт повышения цен. Изменение структуры тарифов добавило меньше рубля, то есть пользователи не стали заметно чаще выбирать более дорогой тариф. Поэтому в дальнейшем анализе рост за счет цены можно связывать именно с новыми тарифами, а не с изменением предпочтений клиентов

In [ ]:
# График 2. Каскад изменений январь по март 2026, млн руб
# Каскадной диаграммы в seaborn нет, поэтому строю ее
# ax.bar(..., bottom=...) - каждый столбик начинается там где закончился предыдущий

# Стиль (шрифты, сетка, рамки) общий, из sns.set_theme в начале ноутбука
steps = pd.Series({
    "база\nБазовый": eff.loc["Базовый", "active"],
    "база\nРасширенный": eff.loc["Расширенный", "active"],
    "платежей\nна подписку": eff["per_sub"].sum(),
    "цена\nБазовый": eff.loc["Базовый", "price"],
    "цена\nРасширенный": eff.loc["Расширенный", "price"],
}) / 1e6
accent = ["цена\nБазовый", "цена\nРасширенный"]

before = steps.cumsum() - steps                    # с какого уровня начинается каждый столбик
colors = [ACCENT if name in accent else GRAY for name in steps.index]

fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
bars = ax.bar(steps.index, steps, bottom=before, color=colors, zorder=2)
total_bar = ax.bar("итого", steps.sum(), color=DARK, zorder=2)
fmt = lambda v: f"{v:+.2f}".replace(".", ",").replace("-", "−")
ax.bar_label(bars, fmt=fmt, padding=3)
ax.bar_label(total_bar, fmt=fmt, padding=3)
ax.axhline(0, color=DARK, linewidth=0.8)
ax.set_ylabel("млн руб.")
ax.margins(y=0.15)
ax.set_title(f"Выручка выросла на {mln(delta)} млн руб., из них {mln(price_effect)} млн ({pct(price_effect / delta, 0)}) дало повышение цен",
             loc="left", fontsize=12, fontweight="bold")
ax.set_xlabel("Январь 2026 -> март 2026 (оба по 31 дню). Порядок подстановок: активные -> платежей на подписку -> цена. "
              "Сумма столбиков = итого.", fontsize=9, color=DARK)
plt.show()

График 2
С января по март, если сравнивать месяцы одинаковой длины, основная часть прироста выручки объясняется повышением цен. Оба тарифа подорожали на 50 рублей, поэтому каждый успешный платеж стал приносить больше. Рост числа подписок тоже повлиял на выручку, но в меньшей степени. Частота платежей на одну подписку практически не изменилась. Таким образом, гипотеза Г2 подтвердилась.

Чтобы не переоценить эффект повышения цен, провожу такую же декомпозицию за более длинный период: с июля 2025-го по июль 2026-го.

In [ ]:
YA, YB = "2025-07", "2026-07"
eff_y = decompose(YA, YB)
delta_y = tot.loc[YB, "revenue"] - tot.loc[YA, "revenue"]

table_y = eff_y.rename(columns=LABELS).copy()
table_y["всего"] = table_y.sum(axis=1)
table_y.loc["итого"] = table_y.sum()
print(f"июль к июлю: {mln(tot.loc[YA, 'revenue'])} -> {mln(tot.loc[YB, 'revenue'])} млн руб., изменение {mln(delta_y)} млн руб.")
print(f"цена {pct(eff_y['price'].sum() / delta_y, 0)}, база {pct(eff_y['active'].sum() / delta_y, 0)}, "
      f"платежей на подписку {pct(eff_y['per_sub'].sum() / delta_y, 0)}")
(table_y / 1e6).round(3)

За год выручка выросла в первую очередь за счет увеличения числа подписок. Повышение цен тоже дало заметный прирост, но его вклад оказался меньше. То есть рост цен дополнил рост базы, а не заменил его.

Теперь важно проверить не привело ли повышение цен к росту оттока и не замедлило ли оно привлечение и удержание подписчиков. 

## 7. Чем заплатили за повышение (Г3-Г9) и что оно изменило в маркетинге (Г10-Г11)

Сравниваю три окна: до (сентябрь 2025 - февраль 2026), после (март - июль 2026) и те же месяцы год назад (март - июль 2025), чтобы отделить эффект цены от сезона. Метрики месячные, поэтому беру среднее по месяцам окна.

In [ ]:
WINDOWS = {
    "до: 09.2025-02.2026": ("2025-09", "2026-02"),
    "после: 03-07.2026": ("2026-03", "2026-07"),
    "год назад: 03-07.2025": ("2025-03", "2025-07"),
}


def window_mean(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Среднее по месяцам каждого окна."""
    out = {}
    for name, (a, b) in WINDOWS.items():
        out[name] = df.loc[a:b, cols].mean()
    return pd.DataFrame(out).T


metrics = ["churn", "churn_expensive", "churn_switched", "new_per_day", "ext_share_new", "ext_share_active", "fail_rate"]
summary = window_mean(tot, metrics)

# Отток по тарифам отдельно (у Базового цена выросла на 25 %, у Расширенного на 12,5 %) 
for t, name in TARIFF.items():
    by_t = window_mean(m.xs(t, level="tariff_id"), ["churn", "churn_expensive", "fail_rate"])
    summary[f"churn {name}"] = by_t["churn"]
    summary[f"churn_expensive {name}"] = by_t["churn_expensive"]
    summary[f"fail_rate {name}"] = by_t["fail_rate"]

summary.T.round(4)

In [ ]:
# насколько изменился отток базового (+25 %) по сравнению с изменением оттока расширенного (+12,5 %)
# если клиенты чувствительны к цене - у Базового отток должен вырасти сильнее
s = summary
did = {}
for col in ["churn", "churn_expensive"]:
    d_basic = s.loc["после: 03-07.2026", f"{col} Базовый"] - s.loc["до: 09.2025-02.2026", f"{col} Базовый"]
    d_ext = s.loc["после: 03-07.2026", f"{col} Расширенный"] - s.loc["до: 09.2025-02.2026", f"{col} Расширенный"]
    did[col] = {"изменение Базовый, п.п.": d_basic * 100, "изменение Расширенный, п.п.": d_ext * 100,
                "разность разностей, п.п.": (d_basic - d_ext) * 100}
pd.DataFrame(did).T.round(3)

In [ ]:
# 95 % доверительный интервал для разницы оттока после - до

# Отток окна считаю по всем месяцам сразу: все ушедшие / все активные на 1-е число

# Если интервал включает ноль, значимой разницы нет, а его верхняя граница показывает рост оттока я могла пропустить
def diff_ci(x1: float, n1: float, x0: float, n0: float) -> tuple[float, float, float]:
    """Разница долей x1/n1 - x0/n0 и 95 % интервал для нее."""
    p1, p0 = x1 / n1, x0 / n0
    se = math.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return p1 - p0, p1 - p0 - 1.96 * se, p1 - p0 + 1.96 * se


def pooled(df: pd.DataFrame, window: str, num: str, den: str) -> tuple[float, float]:
    a, b = WINDOWS[window]
    return df.loc[a:b, num].sum(), df.loc[a:b, den].sum()


BEFORE, AFTER, LY = list(WINDOWS)
rows = []
checks = [("отток, всего", tot, "churned", "active_start"),
          ("отток «дорого»", tot, "churned_expensive", "active_start"),
          ("отток Базовый (+25 %)", m.xs(1, level="tariff_id"), "churned", "active_start"),
          ("отток Расширенный (+12,5 %)", m.xs(2, level="tariff_id"), "churned", "active_start"),
          ("неуспешные списания", tot.assign(attempts=tot["n_failed"] + tot["n_payments"]), "n_failed", "attempts")]
for name, df, num, den in checks:
    for base_name, base in [("до", BEFORE), ("год назад", LY)]:
        x1, n1 = pooled(df, AFTER, num, den)
        x0, n0 = pooled(df, base, num, den)
        d, lo, hi = diff_ci(x1, n1, x0, n0)
        rows.append({"метрика": name, "сравниваем с": base_name, "после": x1 / n1, "база": x0 / n0,
                     "разница, п.п.": d * 100, "95 % ДИ от, п.п.": lo * 100, "95 % ДИ до, п.п.": hi * 100})
ci = pd.DataFrame(rows).set_index(["метрика", "сравниваем с"])
ci.round({"после": 4, "база": 4, "разница, п.п.": 3, "95 % ДИ от, п.п.": 3, "95 % ДИ до, п.п.": 3})

Примечание к таблице - если доверительный интервал включает ноль, статистически значимого изменения оттока не обнаружено. По верхней границе можно оценить, насколько сильный рост оттока мы могли не заметить. При этом подписки, попадающие в несколько месяцев, не независимы (!!) друг от друга, поэтому интервал может быть немного уже, чем должен быть.

Также в периоде до средняя длина месяца составляет 30,2 дня, а в периоде после 30,6 дней. В более длинном месяце у подписок чуть больше времени, чтобы закончиться, поэтому отток после повышения цен может выглядеть немного выше. Это скорее мешает подтвердить отсутствие роста оттока, чем помогает, поэтому вывод можно считать правильным.

In [ ]:
# График 3. Отток по тарифам по месяцам: всего и по причине «дорого» 

# таблица: одна строка = месяц x тариф. hue="тариф" рисует две линии:
# базовый акцентным цветом, расширенный серым, подписи у линий
plot3 = m[["churn", "churn_expensive"]].reset_index()
plot3["date"] = pd.PeriodIndex(plot3["month"], freq="M").to_timestamp()
plot3["тариф"] = plot3["tariff_id"].map(TARIFF)
plot3["churn_pct"] = plot3["churn"] * 100
plot3["churn_expensive_pct"] = plot3["churn_expensive"] * 100
TARIFF_PALETTE = {"Базовый": ACCENT, "Расширенный": GRAY}
TARIFF_RISE = {"Базовый": "+25 %", "Расширенный": "+12,5 %"}

churn_before = summary.loc["до: 09.2025-02.2026", "churn"]
churn_after = summary.loc["после: 03-07.2026", "churn"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")
for ax, col, title in [(axes[0], "churn_pct", "Отток, % активных на начало месяца"),
                       (axes[1], "churn_expensive_pct", "Из них отменили с причиной «дорого», %")]:
    sns.lineplot(data=plot3, x="date", y=col, hue="тариф", palette=TARIFF_PALETTE,
                 marker="o", markersize=5, legend=False, ax=ax)
    # подпись у последней точки каждой линии; если точки рядом, развожу подписи по вертикали
    last = plot3[plot3["date"] == plot3["date"].max()].set_index("тариф")[col]
    close = abs(last["Базовый"] - last["Расширенный"]) < 0.15 * plot3[col].max()
    for name, value in last.items():
        dy = 0 if not close else (7 if value >= last.drop(name).iloc[0] else -7)
        ax.annotate(f"{name} ({TARIFF_RISE[name]})", (plot3["date"].max(), value), xytext=(6, dy),
                    textcoords="offset points", color=TARIFF_PALETTE[name], va="center", fontsize=9)
    ax.axvline(pd.Timestamp("2026-03-01"), color=DARK, linestyle=":", linewidth=1)
    ax.text(pd.Timestamp("2026-03-08"), 0.03, "новые цены", color=DARK, fontsize=9,
            transform=ax.get_xaxis_transform())
    ax.set_ylim(bottom=0)
    ax.set(xlabel="", ylabel="%")
    ax.set_title(title, loc="left", fontsize=11)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))

fig.suptitle(f"Отток после повышения не вырос: {pct(churn_before)} в месяц до и {pct(churn_after)} после, "
             "у Базового (+25 %) так же, как у Расширенного",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.text(0.01, -0.04, "Отток месяца = закончились в месяце / активны на 1-е число. Февраль 2025 - июль 2026, сутки по UTC.",
         fontsize=9, color=DARK)
plt.show()

**Что доказывает график 3.** Если бы клиенты реагировали на цену, после 1 марта отток вырос бы, и сильнее всего у базового тарифа, который подорожал на 25 % (микроэкономика). Этого не видно: обе линии колеблются в том же коридоре, что и до повышения, а отмены «дорого» не стали чаще. 

**Г4 и Г5 по таблице выше.** Новых подписок в день после повышения столько же, сколько до него и в том же сезоне год назад (строка `new_per_day`). Доля расширенного среди новых и среди активных подписок не изменилась (`ext_share_new`, `ext_share_active`), хотя базовый тариф подорожал сильнее в процентах. Значит повышение не повлияло негативно на новых клиентов и не сдвинуло выбор к более дешевому тарифу

In [ ]:
# сколько базы можно было потерять прежде чем повышение стало бы убыточным

# Выручка = активные x выручка на активную подписку (ARPU)
# повышение невыгодно если база упала сильнее, чем 1 - ARPU_до / ARPU_после
arpu_before, arpu_after = tot.loc[A, "arpu"], tot.loc[B, "arpu"]
breakeven_loss = 1 - arpu_before / arpu_after

# Сколько денег принесли сами +50 руб. за март - июль 2026 (оба тарифа подорожали ровно на 50 руб.):
# каждый успешный платеж после 1 марта на 50 руб. больше, чем был бы по старой цене
extra = (m.loc["2026-03":"2026-07", "n_payments"] * 50).sum()

print(f"ARPU: {arpu_before:.2f} -> {arpu_after:.2f} руб. на активную подписку в месяц (+{pct(arpu_after / arpu_before - 1)})")
print(f"повышение остается выгодным, пока база падает меньше чем на {pct(breakeven_loss)}")
print(f"доплата за счет новых цен за март - июль 2026: {mln(extra)} млн руб.")

In [ ]:
# LTV до и после повышения : LTV  равно ARPPU / отток

# Все месяцы окна складываю (как дальше для каналов в Г10-Г11):
#   ARPPU  = выручка окна / платящие подписко-месяцы окна
#   отток  = ушедшие за окно / активные на 1-е число, по всем месяцам окна
def window_sum(window: str, cols: list[str]) -> pd.Series:
    a, b = WINDOWS[window]
    return tot.loc[a:b, cols].sum()


rows = {}
for name, window in [("до повышения", BEFORE), ("после повышения", AFTER)]:
    w = window_sum(window, ["revenue", "n_paying", "churned", "active_start"])
    rows[name] = {"ARPPU, руб.": w["revenue"] / w["n_paying"], "отток в месяц": w["churned"] / w["active_start"]}
ltv = pd.DataFrame(rows).T
ltv["средняя жизнь, мес."] = 1 / ltv["отток в месяц"]
ltv["LTV, руб."] = ltv["ARPPU, руб."] / ltv["отток в месяц"]
ltv0, ltv1 = ltv["LTV, руб."].iloc[0], ltv["LTV, руб."].iloc[1]
print(f"LTV подписки: {ltv0:.0f} -> {ltv1:.0f} руб. (+{pct(ltv1 / ltv0 - 1)})")
ltv.round({"ARPPU, руб.": 2, "отток в месяц": 4, "средняя жизнь, мес.": 1, "LTV, руб.": 0})

Отток не изменился, а платеж вырос, поэтому LTV подписки вырос примерно так же, как ARPPU. Для решения о прибыли нужна маржа, а не выручка: из нее вычитаются кешбэк и выплаты партнерам, их в этом проекте не считаю

### Г7. «После повышения рост базы замедлился»

Чистый прирост базы (сколько подписок прибавилось за месяц) после марта меньше чем осенью 2025. На первый взгляд это похоже на эффект цены. Проверяю: прирост падает сам, потому что новых подписок приходит постоянное число, а уходит постоянная доля от растущей базы. 
Тогда прирост должен был снижаться и до повышения, а база стремится к лимиту - новые в месяц / отток

In [ ]:
# Чистый прирост базы за месяц = активные на 1-е число следующего месяца - активные на 1-е число этого
# Последний месяц витрины (июль 2026) без следующего поэтому ряд до июня 2026
growth = tot[["active_start", "new_subs", "churn"]].copy()
growth["net_adds"] = growth["active_start"].shift(-1) - growth["active_start"]
growth = growth.dropna(subset=["net_adds"])

# при постоянном притоке N в месяц и оттоке c база перестает расти, когда c x база = N
new_month = tot.loc["2026-03":"2026-07", "new_per_day"].mean() * 30.4      # средний месяц после повышения
ceiling = new_month / summary.loc[AFTER, "churn"]
share_of_ceiling = tot.loc["2026-07", "active_start"] / ceiling

def mean_net(a: str, b: str) -> float:
    return growth.loc[a:b, "net_adds"].mean()

print(f"чистый прирост в месяц: 03-07.2025 {mean_net('2025-03', '2025-07'):.0f}, "
      f"09.2025-02.2026 {mean_net('2025-09', '2026-02'):.0f}, 03-06.2026 {mean_net('2026-03', '2026-06'):.0f}")
ceiling_txt = f"{ceiling:,.0f}".replace(",", " ")
print(f"потолок базы при нынешних притоке и оттоке: около {ceiling_txt} подписок, сейчас пройдено {pct(share_of_ceiling, 0)}")
growth[["active_start", "new_subs", "churn", "net_adds"]].round(4).tail(10)

In [ ]:
# График 4. Чистый прирост базы по месяцам 
plot5 = growth.reset_index()
plot5["date"] = pd.PeriodIndex(plot5["month"], freq="M").to_timestamp()
plot5["period"] = ["после" if d >= pd.Timestamp("2026-03-01") else "до" for d in plot5["date"]]

# Линейный тренд по месяцам до повышения (numpy.polyfit), беру его до июня 2026
before5 = plot5["period"] == "до"
k, b0 = np.polyfit(plot5.index[before5], plot5.loc[before5, "net_adds"], deg=1)
plot5["trend"] = k * plot5.index + b0

fig, ax = plt.subplots(figsize=(11, 4.4), layout="constrained")
sns.lineplot(data=plot5, x="date", y="trend", color=GRAY, linestyle="--", linewidth=1, ax=ax)
sns.lineplot(data=plot5, x="date", y="net_adds", hue="period", palette=PALETTE, marker="o", legend=False, ax=ax)
ax.axvline(pd.Timestamp("2026-03-01"), color=DARK, linestyle=":", linewidth=1)
ax.text(pd.Timestamp("2026-03-08"), 0.03, "новые цены", color=DARK, fontsize=9, transform=ax.get_xaxis_transform())
ax.text(0.01, 0.06, "пунктир: тренд до повышения, продолжен вперед", color=GRAY, fontsize=9, transform=ax.transAxes)
ax.set_ylim(bottom=0)
ax.set(xlabel="", ylabel="подписок за месяц")
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
ax.set_title("Прирост базы снижается с 2025 года, а не с марта 2026: после повышения он идет по прежнему тренду",
             loc="left", fontsize=12, fontweight="bold")
fig.text(0.01, -0.04, "Чистый прирост = активные на 1-е число следующего месяца - активные на 1-е число этого. Февраль 2025 - июнь 2026.",
         fontsize=9, color=DARK)
plt.show()

**Что доказывает график 4.** Прирост базы снижается весь период, и после 1 марта точки идут по продолжению прежнего тренда, а не ниже него. Это скорее всего потому, что приток постоянный, а отток растет вместе с базой. Г7 не подтвердилась как эффект цены, но дала важный вывод для бизнеса: клиентская база приближается к потолку и дальше расти за счет новых клиентов будет все труднее. Отсюда следует, что главные рычаги выручки дальше это цена и удержание

### Г8. «Отток различается по каналам привлечения»

Чтобы рекомендация была конкретной, нужно понять, где теряются подписки. Канал привлечения возьму у владельца подписки (`clients.acquisition_channel`), клиентов без канала держу отдельной строкой (по той же логике что и семинар 6))

Проверяю ее на всем окне после повышения (март - июль 2026) и сравниваю с окном до

Если гипотеза верна, отток каналов будет заметно различаться, и разница будет видимой

In [ ]:
# Витрина «месяц x канал привлечения» для гипотез Г8-Г11
# subs  - подписка и канал ее владельца. left join с clients: подписка без найденного клиента не потеряется,
#         client_id в clients уникален (PK), поэтому строки подписок не множатся 
#         Пустой канал заменяю на "без канала", чтобы group by его не выбросил
# act   - база как в главной витрине: новые, активные на 1-е число, ушедшие, активные, из них расширенный
# rev   - выручка без двойных списаний и сколько разных подписок заплатили
# spend - маркетинговые расходы канала за месяц (marketing_spend сходится с клиентами по каналу, ключа нет)
by_channel = q("""
    with months as (
        select generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m
    ),
    subs as (
        select s.subscription_id, s.started_at, s.ended_at, s.tariff_id,
               coalesce(c.acquisition_channel, 'без канала') as channel
        from prime.subscriptions s
        left join prime.clients c on c.client_id = s.owner_client_id
    ),
    act as (
        select to_char(mo.m, 'YYYY-MM') as month,
               sb.channel,
               count(*) filter (where sb.started_at >= mo.m)    as new_subs,
               count(*) filter (where sb.started_at < mo.m)     as active_start,
               count(*) filter (where sb.started_at < mo.m
                                  and sb.ended_at < mo.m + interval '1 month') as churned,
               count(*)                                         as active,
               count(*) filter (where sb.tariff_id = 2)         as active_ext
        from months mo
        join subs sb
          on sb.started_at < mo.m + interval '1 month'
         and (sb.ended_at is null or sb.ended_at >= mo.m)
        group by 1, 2
    ),
    ok as (
        select subscription_id, amount, paid_at,
               paid_at - lag(paid_at) over (partition by subscription_id, amount order by paid_at) as gap
        from prime.payments
        where status = 'success'
    ),
    rev as (
        select to_char(ok.paid_at, 'YYYY-MM') as month,
               sb.channel,
               count(distinct ok.subscription_id) as n_paying,
               sum(ok.amount)                     as revenue
        from ok
        join subs sb using (subscription_id)
        where (ok.gap is null or ok.gap >= interval '20 minutes')
          and ok.paid_at >= cast(:first as timestamptz)
          and ok.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    ),
    spend as (
        select to_char(spend_date, 'YYYY-MM') as month, channel, sum(amount) as spend
        from prime.marketing_spend
        where spend_date >= cast(:first as date)
          and spend_date <  cast(:last as date) + interval '1 month'
        group by 1, 2
    )
    select a.*,
           coalesce(r.n_paying, 0) as n_paying,
           coalesce(r.revenue, 0)  as revenue,
           coalesce(sp.spend, 0)   as spend
    from act a
    left join rev r using (month, channel)
    left join spend sp using (month, channel)
    order by 1, 2
""", first=FIRST, last=LAST)

# Сверка: по каналам в сумме столько же активных и ушедших, сколько в витрине (никого не потеряли и не задвоили)
check_ch = by_channel.groupby("month")[["active_start", "churned"]].sum()
assert (check_ch["active_start"] == tot["active_start"]).all(), "каналы не сходятся с витриной по активным"
assert (check_ch["churned"] == tot["churned"]).all(), "каналы не сходятся с витриной по ушедшим"
check_rev = by_channel.groupby("month")[["revenue", "n_paying"]].sum()
assert (check_rev["revenue"] - tot["revenue"]).abs().max() < 0.01, "каналы не сходятся с витриной по выручке"
assert (check_rev["n_paying"] == tot["n_paying"]).all(), "каналы не сходятся с витриной по платящим"

# Расходы: все ли деньги из marketing_spend попали в витрину (вдруг в расходах есть канал, которого нет у клиентов)
spend_total = q("""
    select sum(amount) as spend
    from prime.marketing_spend
    where spend_date >= cast(:first as date) and spend_date < cast(:last as date) + interval '1 month'
""", first=FIRST, last=LAST).iloc[0, 0]
assert abs(by_channel["spend"].sum() - spend_total) < 0.01, "часть расходов не сопоставилась с каналами клиентов"
print("сверка пройдена: активные, ушедшие, выручка, платящие и расходы по каналам сходятся с итогами;",
      by_channel["channel"].nunique(), "каналов, включая «без канала»")

In [ ]:
# Для каждого канала отток до и после повышения (все подписко-месяцы окна вместе),
# доля канала в базе и в притоке. 
ch = by_channel.set_index(["channel", "month"]).sort_index()
rows = []
for channel in ch.index.get_level_values("channel").unique():
    df = ch.loc[channel]
    x1, n1 = pooled(df, AFTER, "churned", "active_start")
    x0, n0 = pooled(df, BEFORE, "churned", "active_start")
    d, lo, hi = diff_ci(x1, n1, x0, n0)
    rows.append({"канал": channel,
                 "доля в базе": df.loc["2026-07", "active_start"] / tot.loc["2026-07", "active_start"],
                 "новых в месяц (после)": df.loc["2026-03":"2026-07", "new_subs"].mean(),
                 "отток до": x0 / n0, "отток после": x1 / n1,
                 "разница, п.п.": d * 100, "95 % ДИ от": lo * 100, "95 % ДИ до": hi * 100})
churn_ch = pd.DataFrame(rows).set_index("канал").sort_values("отток после", ascending=False)

spread = churn_ch["отток после"].max() / churn_ch["отток после"].min()
spread_txt = f"{spread:.1f}".replace(".", ",")
print(f"Г8. Отток после повышения: от {pct(churn_ch['отток после'].min())} до {pct(churn_ch['отток после'].max())} в месяц, "
      f"разница в {spread_txt} раза")
churn_ch[["доля в базе", "новых в месяц (после)", "отток до", "отток после"]].round(
    {"доля в базе": 3, "новых в месяц (после)": 0, "отток до": 4, "отток после": 4})

#Разница "после - до" и ее 95 % ДИ нужны для Г9 ниже

In [ ]:
# Сколько базы даст удержание в худших каналах
# Потолок базы по каналу = новых в месяц / отток канала (как в Г7, но для каждого канала отдельно)
# Сценарий: два канала с самым высоким оттоком снижают его до медианы по каналам
churn_ch["потолок базы"] = churn_ch["новых в месяц (после)"] / churn_ch["отток после"]
median_churn = churn_ch["отток после"].median()
worst = churn_ch.index[:2].tolist()

scenario = churn_ch["отток после"].copy()
scenario[worst] = median_churn
ceiling_now = churn_ch["потолок базы"].sum()
ceiling_new = (churn_ch["новых в месяц (после)"] / scenario).sum()

print("каналы с самым высоким оттоком:", ", ".join(worst))
print(f"медиана оттока по каналам: {pct(median_churn)} в месяц")
now_txt, new_txt = f"{ceiling_now:,.0f}".replace(",", " "), f"{ceiling_new:,.0f}".replace(",", " ")
print(f"потолок базы: {now_txt} -> {new_txt} подписок (+{pct(ceiling_new / ceiling_now - 1)}), "
      "если в этих двух каналах отток снизится до медианы")
churn_ch[["доля в базе", "отток после", "потолок базы"]].round({"доля в базе": 3, "отток после": 4, "потолок базы": 0})

**Г8 подтвердилась.** Отток сильно различается по каналам: у organic около 2 % в месяц, у social около 7 %, у performance около 5,5 %. Разница устойчивая:порядок каналов до и после почти одинаковый, и после повышения. Теперь можно перейти к конкретным рекомендациям по удержанию: определить, в каких каналах стоит сосредоточиться на работе с оттоком и какого эффекта это позволит добиться

Потолок здесь (309 тыс.) выше, чем в Г7 (280 тыс.), потому что считается по каждому каналу отдельно: у каналов разный отток, и в долгосрочной базе больше клиентов с низким оттоком (organic). Общая формула в Г7 этого не учитывает и дает оценку снизу. Для вывода важно не точное значение, а то, что база к нему приближается и снижение оттока в social и performance до медианы поднимает потолок примерно на 13 %

### Г9. «Повышение цен сильнее ударило по каналам, чувствительным к цене»

Г3 показала, что отток в целом после повышения не вырос. Но средний отток мог не измениться, если в одних каналах он вырос, а в других упал. Надо проверить каждый канал отдельно: если клиенты какого-то канала чувствительны к цене, у него отток после должен быть выше, чем «до», и 95 % интервал разницы не должен включать ноль

In [ ]:
# Г9. Изменение оттока каждого канала после повышения и 95 % ДИ этой разницы
# Интервал не включает ноль = изменение статистически заметно
g9 = churn_ch[["отток до", "отток после", "разница, п.п.", "95 % ДИ от", "95 % ДИ до"]].copy()
g9["заметно выросло"] = g9["95 % ДИ от"] > 0
print("каналов, где отток после повышения заметно вырос:", int(g9["заметно выросло"].sum()), "из", len(g9))
g9.round({"отток до": 4, "отток после": 4, "разница, п.п.": 3, "95 % ДИ от": 3, "95 % ДИ до": 3})

**Г9 не подтвердилась.** После 1 марта отток заметно не вырос ни в одном канале: разница между периодами составляет лишь десятые доли процентного пункта, а доверительные интервалы включают ноль. 
Ближе всего к границе значимости оказался канал social: отток вырос примерно на 0,2 пп., а нижняя граница интервала находится почти на нуле. За этим каналом стоит понаблюдать внимательнее, но пока нельзя утверждать, что причина именно в повышении цены
При этом у social и performance отток был высоким ещё до марта. Значит повышение цен его не вызвало, а проблему удержания в этих каналах стоит решать отдельно от ценовой политики

### Г10-Г11. Как повышение цен изменило окупаемость маркетинга

Повышение подняло выручку с каждой платящей подписки, а отток не изменился ни в целом (Г3), ни в каналах (Г9). Отсюда вывод, что вырос LTV подписки, и окупаемость каналов привлечения нужно пересчитать.
CAC = расходы канала / новые подписки канала, LTV = ARPPU / отток, порог окупаемости LTV/CAC = 3. Новое по сравнению с семинаром (там мы то же самое считали): сравниваю окна до и после повышения, раскладываю изменение окупаемости на факторы и смотрю, куда идет бюджет.

In [ ]:
# Юнит-экономика каналов в окнах «до» и «после». Все месяцы окна складываются (раздел 4)
def unit_economics(window: str) -> pd.DataFrame:
    a, b = WINDOWS[window]
    in_window = by_channel[(by_channel["month"] >= a) & (by_channel["month"] <= b)]
    n_months = in_window["month"].nunique()
    w = in_window.groupby("channel")[["churned", "active_start", "revenue", "n_paying",
                                      "spend", "new_subs", "active", "active_ext"]].sum()
    u = pd.DataFrame(index=w.index)
    u["доля бюджета"] = w["spend"] / w["spend"].sum()
    u["расходы в месяц, млн руб."] = w["spend"] / n_months / 1e6
    u["ARPPU"] = w["revenue"] / w["n_paying"]                    # выручка на платящую подписку в месяц
    u["доля Расширенного"] = w["active_ext"] / w["active"]
    u["отток"] = w["churned"] / w["active_start"]
    u["CAC"] = (w["spend"] / w["new_subs"]).where(w["spend"] > 0)   # у «без канала» расходов нет (CAC не считаем)
    u["LTV"] = u["ARPPU"] / u["отток"]
    u["LTV/CAC"] = u["LTV"] / u["CAC"]
    return u


ue_before, ue_after = unit_economics(BEFORE), unit_economics(AFTER)
VERDICT = dict(bins=[0, 1, 3, np.inf], labels=["убыточен", "на грани", "окупается"])   # правило семинара 5

ue = pd.DataFrame({
    "доля бюджета (после)": ue_after["доля бюджета"],
    "CAC до": ue_before["CAC"], "CAC после": ue_after["CAC"],
    "отток до": ue_before["отток"], "отток после": ue_after["отток"],
    "ARPPU до": ue_before["ARPPU"], "ARPPU после": ue_after["ARPPU"],
    "LTV до": ue_before["LTV"], "LTV после": ue_after["LTV"],
    "LTV/CAC до": ue_before["LTV/CAC"], "LTV/CAC после": ue_after["LTV/CAC"],
})
ue["вердикт до"] = pd.cut(ue["LTV/CAC до"], **VERDICT)
ue["вердикт после"] = pd.cut(ue["LTV/CAC после"], **VERDICT)
ue = ue.sort_values("LTV/CAC после")

ltv_growth = ue["LTV после"] / ue["LTV до"] - 1
print(f"LTV каналов после повышения изменился на {ltv_growth.min() * 100:+.0f} ... {ltv_growth.max() * 100:+.0f} %")
ue.round({"доля бюджета (после)": 3, "CAC до": 0, "CAC после": 0, "отток до": 4, "отток после": 4,
          "ARPPU до": 1, "ARPPU после": 1, "LTV до": 0, "LTV после": 0, "LTV/CAC до": 2, "LTV/CAC после": 2})

#### Г10. «Дорогой канал приводит более ценных клиентов»

Самый дорогой канал по CAC это performance. Частое объяснение высокой цены привлечения: платный трафик приводит более ценных клиентов, которые больше платят (или дольше живут). 
Проверю на окне после повышения: если гипотеза верна, у performance выше ARPPU или доля расширенного, или ниже отток, чем у других каналов

In [ ]:
# Г10. Performance против остальных платных каналов (окно после повышения)
paid = ue_after[ue_after["CAC"].notna()]
others = paid.drop(index="performance")
g10 = pd.DataFrame({
    "performance": paid.loc["performance", ["ARPPU", "доля Расширенного", "отток", "CAC"]],
    "остальные каналы, от": others[["ARPPU", "доля Расширенного", "отток", "CAC"]].min(),
    "остальные каналы, до": others[["ARPPU", "доля Расширенного", "отток", "CAC"]].max(),
    "organic": paid.loc["organic", ["ARPPU", "доля Расширенного", "отток", "CAC"]],
})
g10["performance / organic"] = g10["performance"] / g10["organic"]
g10.round(4)

**Г10 не подтвердилась.** Клиенты performance платят столько же, сколько клиенты других каналов, и также часто выбирают расширенный тариф: ARPPU и тарифы у всех каналов одинаковые. При этом они уходят заметно чаще organic, а привлечение стоит примерно в 11 раз дороже (строка `performance / organic`). Высокий CAC не компенсируется ценностью клиента (дорогой канал приводит не лучших, а обычных клиентов, которые к тому же быстрее уходят)

#### Г11. «Новая цена сделала все каналы окупаемыми»

Если гипотеза верна, у всех каналов после повышения LTV/CAC не ниже 3. Изменение окупаемости каждого канала раскладываю на факторы. LTV/CAC = ARPPU / отток / CAC, а цепные подстановки работают с произведением, поэтому делители переписываю в множители:

> **LTV/CAC = подписок на рубль (1 / CAC) x срок жизни (1 / отток) x ARPPU**

* подписок на рубль - сколько новых подписок приносит рубль маркетинга (эффективность привлечения)
* срок жизни -сколько месяцев в среднем живет подписка (удержание)
* ARPPU -сколько платящая подписка платит в месяц 

Длина месяцев в окнах разная (30,2 и 30,6 дня), но ARPPU и отток посчитаны за месяц и сдвигаются от этого в одну сторону, поэтому на их отношение, LTV, это почти не влияет

In [ ]:
# Г11. Разложение изменения LTV/CAC каждого платного канала: до - после повышения
F2 = ["per_rub", "life", "arppu"]
LABELS2 = {"per_rub": "подписок на рубль (1 / CAC)", "life": "срок жизни (1 / отток)", "arppu": "ARPPU"}


def ltv_factors(u: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({"per_rub": 1 / u["CAC"], "life": 1 / u["отток"], "arppu": u["ARPPU"]})


fb, fa = ltv_factors(ue_before), ltv_factors(ue_after)
paid_channels = ue_after.index[ue_after["CAC"].notna()]
rows = {}
for c in paid_channels:
    eff = contrib(fb.loc[c].to_dict(), fa.loc[c].to_dict(), F2)      # внутри contrib стоит assert на сумму влияний
    rows[c] = {"LTV/CAC до": ue_before.loc[c, "LTV/CAC"], "LTV/CAC после": ue_after.loc[c, "LTV/CAC"], **eff}
g11 = pd.DataFrame(rows).T
g11["изменение"] = g11["LTV/CAC после"] - g11["LTV/CAC до"]
assert (g11[F2].sum(axis=1) - g11["изменение"]).abs().max() < 1e-6, "сумма влияний не равна изменению LTV/CAC"
# влияния показываю в единицах LTV/CAC, а не в долях: когда факторы тянут в разные стороны,
# доли выходят за 100 % и только путают (лекция 6)

# зависит ли вывод от порядка подстановок: влияние ARPPU во всех 6 порядках
arppu_by_order = pd.DataFrame({
    " -> ".join(order): {c: contrib(fb.loc[c].to_dict(), fa.loc[c].to_dict(), list(order))["arppu"] for c in paid_channels}
    for order in itertools.permutations(F2)})
g11["ARPPU: мин. по порядкам"] = arppu_by_order.min(axis=1)
g11["ARPPU: макс. по порядкам"] = arppu_by_order.max(axis=1)

g11.rename(columns=LABELS2).sort_values("LTV/CAC после").round(3)

In [ ]:
# какой CAC или какой отток нужны, чтобы при нынешних ARPPU выйти на LTV/CAC = 3

#   CAC для порога   = LTV / 3
#   отток для порога = ARPPU / (3 x CAC)
below = ue_after[ue_after["LTV/CAC"] < 3]
need = pd.DataFrame({
    "LTV/CAC": below["LTV/CAC"],
    "доля бюджета": below["доля бюджета"],
    "расходы в месяц, млн руб.": below["расходы в месяц, млн руб."],
    "CAC сейчас": below["CAC"],
    "CAC для порога 3": below["LTV"] / 3,
    "отток сейчас": below["отток"],
    "отток для порога 3": below["ARPPU"] / (3 * below["CAC"]),
})
need["снизить CAC на"] = 1 - need["CAC для порога 3"] / need["CAC сейчас"]
need["снизить отток на, п.п."] = (need["отток сейчас"] - need["отток для порога 3"]) * 100
budget_below = below["доля бюджета"].sum()
spend_below = below["расходы в месяц, млн руб."].sum()

spend_below_txt = f"{spend_below:.1f}".replace(".", ",")
print(f"каналов ниже порога после повышения: {len(below)}; на них {pct(budget_below, 0)} бюджета, "
      f"около {spend_below_txt} млн руб. в месяц")
need.round({"LTV/CAC": 2, "доля бюджета": 3, "расходы в месяц, млн руб.": 2, "CAC сейчас": 0, "CAC для порога 3": 0,
            "отток сейчас": 4, "отток для порога 3": 4, "снизить CAC на": 3, "снизить отток на, п.п.": 2})

In [ ]:
# График 5. Окупаемость каналов до и после повышения цен 
# Светлая точка до повышения, яркая - после, каналы ниже порога после повышения - акцентным цветом
# Шкала логарифмическая: у каналов LTV/CAC от ~3 до ~80
plot5 = pd.DataFrame({"до": ue_before["LTV/CAC"], "после": ue_after["LTV/CAC"],
                      "доля бюджета": ue_after["доля бюджета"]}).dropna().sort_values("после")
plot5["y"] = range(len(plot5))
plot5["цвет"] = [ACCENT if v < 3 else GRAY for v in plot5["после"]]

fig, ax = plt.subplots(figsize=(11, 4.8), layout="constrained")
ax.hlines(plot5["y"], plot5["до"], plot5["после"], colors=plot5["цвет"], linewidth=2, zorder=1)
palette5 = dict(zip(plot5.index, plot5["цвет"]))
sns.scatterplot(data=plot5.reset_index(names="канал"), x="до", y="y", hue="канал", palette=palette5,
                s=60, alpha=0.35, legend=False, ax=ax, zorder=2)
sns.scatterplot(data=plot5.reset_index(names="канал"), x="после", y="y", hue="канал", palette=palette5,
                s=80, legend=False, ax=ax, zorder=3)
for name, row in plot5.iterrows():
    ax.annotate(f"{row['до']:.1f} -> {row['после']:.1f}".replace(".", ","), (row["после"], row["y"]),
                xytext=(8, 0), textcoords="offset points", va="center", fontsize=9, color=row["цвет"])
ax.axvline(3, color=DARK, linestyle="--", linewidth=1)
ax.set_ylim(-1, len(plot5) - 0.5)
ax.text(3.1, -0.75, "порог окупаемости 3", color=DARK, fontsize=9)
ax.set_xscale("log")
lo, hi = plot5[["до", "после"]].min().min(), plot5[["до", "после"]].max().max()
ax.set_xlim(min(1, lo * 0.8), max(200, hi * 1.5))
ticks = [t for t in [0.5, 1, 2, 3, 5, 10, 20, 50, 100, 200] if ax.get_xlim()[0] <= t <= ax.get_xlim()[1]]
ax.set_xticks(ticks, labels=[f"{t:g}".replace(".", ",") for t in ticks])
ax.set_yticks(plot5["y"], labels=[f"{n} ({pct(sh, 0)} бюджета)" for n, sh in plot5["доля бюджета"].items()])
ax.set(xlabel="LTV / CAC, логарифмическая шкала", ylabel="")
ax.grid(axis="x", color="#E5E7EB")
ax.grid(axis="y", visible=False)

if len(below):
    title = (f"{pct(budget_below, 0)} маркетингового бюджета уходит в канал ниже порога окупаемости: "
             + ", ".join(f"{c} {ue_before.loc[c, 'LTV/CAC']:.1f} -> {ue_after.loc[c, 'LTV/CAC']:.1f}".replace(".", ",") for c in below.index))
else:
    title = "После повышения цен все каналы выше порога окупаемости 3"
fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.text(0.01, -0.05, f"LTV каналов после повышения изменился на {ltv_growth.min() * 100:+.0f} ... {ltv_growth.max() * 100:+.0f} %. "
         "LTV/CAC = (ARPPU / отток) / CAC; окна 09.2025-02.2026 и 03-07.2026; доля бюджета в окне после. "
         "«Без канала» нет расходов, CAC не считается.", fontsize=9, color=DARK)
plt.show()

**Что доказывает график 5.** После повышения цен LTV/CAC вырос во всех каналах: подписка стала приносить больше денег, а срок ее жизни почти не изменился. Но performance, на который приходится половина маркетингового бюджета, все еще не достиг порога окупаемости: показатель вырос с 2,3 до 2,8 при пороге 3. Разложение показывает, за счет чего произошел рост: почти весь прирост LTV/CAC обеспечило повышение среднего дохода с подписки, ARPPU (+0,45 из +0,47). Удержание и стоимость привлечения не улучшились.

В таблице «каналы ниже порога» показано, сколько не хватает до окупаемости: снизить CAC на 6,4 % (с 2 157 до 2 018 руб.) или отток на 0,35 п.п. (с 5,47 до 5,12 % в месяц).

**Г11 подтвердилась частично.** Окупаемость выросла у всех каналов, но не у всех до порога 3. Новая цена улучшила экономику привлечения, но проблему дорогого канала не решила: ее нужно решать удержанием или стоимостью привлечения, а не ценой.

### Итог по гипотезам

| | Гипотеза | Результат | Почему |
|---|---|---|---|
| Г1 | Прирост дал один канал продаж | не подтвердилась | все семь каналов выросли одинаково, каждый дал около 1/7 прироста |
| Г2 | Прирост в марте дала цена | подтвердилась | в декомпозиции январь по март цена главный фактор при любом порядке подстановок |
| Г3 | Вырос отток, особенно «дорого» | не подтвердилась | отток после марта на уровне «до» и того же сезона год назад; у Базового (+25 %) не сильнее, чем у Расширенного (+12,5 %) |
| Г4 | Упал приток новых подписок | не подтвердилась | новых подписок в день столько же, сколько до повышения |
| Г5 | Клиенты ушли на дешевый тариф | не подтвердилась | доля Расширенного среди новых и среди активных не изменилась, отмены «перешел» на прежнем уровне |
| Г6 | Стало больше неуспешных списаний | не подтвердилась | доля failed та же, что до повышения |
| Г7 | После повышения замедлился рост базы | не подтвердилась как эффект цены | прирост снижается с 2025 года по механике постоянный приток, постоянная доля оттока; база идет к потолку |
| Г8 | Отток различается по каналам привлечения | подтвердилась | от ~2 % в месяц у organic до ~7 % у social; порядок каналов одинаковый до и после |
| Г9 | Повышение сильнее ударило по каналам, чувствительным к цене | не подтвердилась | ни в одном канале отток после повышения заметно не вырос, интервалы разницы включают ноль |
| Г10 | Дорогой канал (performance) приводит более ценных клиентов | не подтвердилась | ARPPU и доля Расширенного как у других каналов, отток выше, CAC примерно в 11 раз выше, чем у organic |
| Г11 | Новая цена сделала все каналы окупаемыми | подтвердилась частично | LTV/CAC вырос у всех каналов, почти целиком за счет ARPPU; но канал с самой большой долей бюджета остался ниже порога 3 (график 5) |

In [ ]:
# Ключевые числа для вывода
key = pd.Series({
    "выручка январь 2026, млн руб.": tot.loc[A, "revenue"] / 1e6,
    "выручка март 2026, млн руб.": tot.loc[B, "revenue"] / 1e6,
    "изменение, млн руб.": delta / 1e6,
    "из них цена, млн руб.": price_effect / 1e6,
    "из них база, млн руб.": base_effect / 1e6,
    "доля цены в приросте": price_effect / delta,
    "доля цены за год (июль к июлю)": eff_y["price"].sum() / delta_y,
    "доплата от новых цен 03-07.2026, млн руб.": extra / 1e6,
    "отток до, в месяц": summary.loc[BEFORE, "churn"],
    "отток после, в месяц": summary.loc[AFTER, "churn"],
    "отток год назад, в месяц": summary.loc[LY, "churn"],
    "верхняя граница роста оттока, п.п.": ci.loc[("отток, всего", "до"), "95 % ДИ до, п.п."],
    "новых в день до": summary.loc[BEFORE, "new_per_day"],
    "новых в день после": summary.loc[AFTER, "new_per_day"],
    "запас прочности (можно потерять базы)": breakeven_loss,
    "LTV до, руб.": ltv0,
    "LTV после, руб.": ltv1,
    "потолок базы, подписок": ceiling,
    "отток по каналам: минимум": churn_ch["отток после"].min(),
    "отток по каналам: максимум": churn_ch["отток после"].max(),
    "потолок базы, если худшие 2 канала до медианы": ceiling_new,
    "рост LTV каналов: минимум": ltv_growth.min(),
    "рост LTV каналов: максимум": ltv_growth.max(),
    "каналов ниже порога 3 после повышения": len(below),
    "их доля бюджета": budget_below,
    "их расходы в месяц, млн руб.": spend_below,
    "нужно снизить CAC на (максимум по ним)": need["снизить CAC на"].max() if len(need) else float("nan"),
    "или отток на, п.п. (максимум по ним)": need["снизить отток на, п.п."].max() if len(need) else float("nan"),
})
key.round(4)

## 8. Вывод и рекомендация (все числа ниже из ячейки «ключевые числа» и таблиц выше)

**Что случилось.** 1 марта 2026 абонентская плата выросла на 50 руб. в обоих тарифах (Базовый +25 %, Расширенный +12,5 %). Выручка марта 2026 составила 58,2 млн руб., на 11,3 млн (24 %) больше январской; оба месяца по 31 дню. Выручка в день выросла скачком на 21 % после месяцев ровного роста около 3 % в месяц (график 1).

**За счет чего.** 8,95 млн из 11,3 млн прироста (79 %) дала цена, 2,3 млн рост базы (график 2); вывод не зависит от порядка подстановок. За год (июль к июлю) главный источник роста по-прежнему база: 60 % против 39 % у цены. За март - июль новые цены принесли около 45,6 млн руб. Платить за повышение не пришлось: отток 3,95 % в месяц до и 3,93 % после (95 % ДИ разницы от -0,07 до +0,04 п.п.) и ниже, чем в том же сезоне год назад (4,11 %); приток, выбор тарифа и неуспешные списания тоже не изменились, и ни в одном канале привлечения отток не вырос (Г3-Г6, Г9, график 3). Рост базы замедляется, но не из-за цены: база идет к потолку около 280 тыс. подписок и прошла уже две трети пути (Г7, график 4).

**Что это изменило в маркетинге.** LTV подписки вырос с 7 078 до 8 432 руб. (+19 %), по каналам от +15 до +26 %, и окупаемость всех каналов улучшилась (график 5). Но performance, на который уходит половина маркетингового бюджета (5,3 млн руб. в месяц), остался ниже порога окупаемости: LTV/CAC 2,3 -> 2,8 при пороге 3. Его клиенты не ценнее других (Г10): ARPPU такой же, как у organic, а отток в 2,5 раза выше. Почти весь прирост его окупаемости дала цена (+0,45 из +0,47), стоимость привлечения и удержание не изменились. До порога не хватает немного: снизить CAC на 6,4 % (с 2 157 до 2 018 руб.) или отток на 0,35 п.п. (с 5,47 до 5,12 % в месяц).

**Чего я не знаю.** Повышение получили все клиенты сразу, контрольной группы нет: «после» сравниваем с «до» и с тем же сезоном год назад. Прошло 5 месяцев, на годовом горизонте отток может отреагировать позже. LTV посчитан по выручке и по формуле: с маржой (минус кешбэк и выплаты партнерам) окупаемость всех каналов ниже, и performance дальше от порога. Неизвестно, как вырастет CAC других каналов, если дать им больше денег. Август 2026 не анализировала из-за пакетной отмены partner_telecom 

**Что делать.**
1. Новые цены сохранить. Повышение принесло дополнительную выручку без измеримых потерь клиентов. Следующее повышение проверять пилотом с контрольной группой, а не на всей базе.
2. Маркетингу, в течение квартала: вывести performance на порог окупаемости. Цель: снизить его CAC с 2 157 до 2 018 руб. (-6,4 %) за счет пересмотра площадок и ставок или его отток с 5,47 до 5,12 % в месяц. Ежемесячно пересчитывать LTV/CAC канала по методике этого ноутбука.
3. Если за квартал цель не достигнута, не наращивать бюджет performance и перевести часть денег в каналы с запасом окупаемости (crm, referral, partner_telecom), сначала пилотом на небольшой сумме, чтобы увидеть, как растет их CAC с бюджетом.
4. Команде удержания начать с каналов с самым высоким оттоком (social и performance). Если их отток снизится до медианы по каналам (3,8 %), потолок базы вырастет примерно на 13 %. Г7 объясняет, почему удержание дальше главный рычаг роста.
5. Владельцам данных: удалить пересекающуюся строку 429 руб. из `prime.tariffs` и вернуть клиентам двойные списания (раздел 3).